In [ ]:
import urllib.request
import datetime
import time
import json
import pandas as pd

ServiceKey = '119fab7a14e0db0b2df8bea880e44b7bda6f4457d4dd64dd3671c8595c5fe276'

In [13]:
#[CODE 1]
def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for Url: %s" % (datetime.datetime.now(), url))
        return None

In [20]:
#[CODE 2]
def getResponse(searchTypeCd, regionCd, numOfRows, pageNo):
    service_url = "http://apis.data.go.kr/B500001/myportal/travel/travellist"
    parameters = "searchTypeCd=" + searchTypeCd
    parameters += "&regionCd=" + regionCd
    parameters += "&numOfRows=" + str(numOfRows)
    parameters += "&pageNo=" + str(pageNo)
    parameters += "&serviceKey=" + ServiceKey #인증키
    parameters += "&_type=json"
###형태: http://apis.data.go.kr/B500001/myportal/travel/travellist?searchTypeCd=01&regionCd=HA&numOfRows=10&pageNo=1&serviceKey=인증키(URL Encode)&_type=xml###
    url = service_url + '?' + parameters
    
    print(url)  #액세스 거부 여부 확인용 출력기
    responseDecode = getRequestUrl(url)  #[CODE 1]
    
    if (responseDecode == None):
        return None
    else:
        print(responseDecode)
        return json.loads(responseDecode)

In [ ]:
#[CODE 3]
def getService(searchTypeCd, regionCd, numOfRows, pageNo):
    jsonResult = []
    cnt = 0
    jsonData = getResponse(searchTypeCd, regionCd, numOfRows, pageNo) #[CODE 2]
    
    while True:
        # API 실패 → 종료
        if jsonData is None:
            break
        
        # 데이터 없음 → 종료
        if jsonData['response'].get('body', {}).get('items') is None:
            break
        
        cnt += 1
        jsonData = getResponse(searchTypeCd, regionCd, numOfRows, pageNo) #[CODE 2]
        region = jsonData['response']['body']['items']['item']['region']
        title = jsonData['response']['body']['items']['item']['title']
        intro = jsonData['response']['body']['items']['item']['intro']
        course = jsonData['response']['body']['items']['item']['course']

        jsonResult.append({'count':cnt, 'searchTypeCd':searchTypeCd, 'regionCd':regionCd, 'region': region, 'title':title, 'intro':intro, 'course':course, 'numOfRows':numOfRows, 'pageNo':pageNo})

        # 다음 페이지로 이동
        pageNo += 1
        jsonData = getResponse(searchTypeCd, regionCd, numOfRows, pageNo)
    
    return jsonResult, searchTypeCd, regionCd, numOfRows, pageNo

In [ ]:
#[CODE 0]
def main():
    result = []
    searchTypeCd = input("(01, 02)>> ")
    regionCd = input("(HA,GU,ND,SJ,YS, 미입력)>> ")
    numOfRows = 10
    pageNo = 1
    
    result, searchTypeCd, regionCd, numOfRows, pageNo = getService(searchTypeCd, regionCd, numOfRows, pageNo)
    
    with open('./%s_%s.json' % (result[0]['region'], result[0]['title']), 'w', encoding='utf8') as outfile:
            jsonFile = json.dumps(result, indent = 4, sort_keys = True, ensure_ascii = False)
            outfile.write(jsonFile)
            
if __name__=='__main__':
    main()

http://apis.data.go.kr/B500001/myportal/travel/travellist?searchTypeCd=01&regionCd=HA&numOfRows=10&pageNo=1&serviceKey=119fab7a14e0db0b2df8bea880e44b7bda6f4457d4dd64dd3671c8595c5fe276&_type=json
type object 'datetime.datetime' has no attribute 'datetime'


AttributeError: type object 'datetime.datetime' has no attribute 'datetime'